## Install Dependencies


In [1]:
# Dependencis for the notebook
%pip install spacy pandas torch transformers huggingface_hub matplotlib ipympl rdflib sentence_transformers numpy
# optuna and optional dependencies
%pip install optuna scikit-learn plotly 
# Qwen optimization (optional) dependencies
%pip install flash-linear-attention causal-conv1d
# Mistral dependencies
%pip install accelerate


/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib.pyplot as plt
import matplotlib
import optuna.visualization
import optuna.importance
import json
from pathlib import Path
from collections import OrderedDict
import modules.llms as llm_parsers
import modules.deepparse_parser as deepparse
import modules.libpostal_client as libpostal_client
import modules.token_classifiers as token_classifiers
from modules.utils import compare_preds, format_time, natural_casing
from typing import NamedTuple
from tqdm.auto import tqdm

from typing import Callable

import abc
import contextlib
import pandas as pd
import time
import pprint
import textwrap
import numpy as np
import modules.train_ner as train_ner
import re

# Total number of addresses in the complete BZK data for calculating estimated time of processing
TOTAL_ADDRESSES = 4_394_539

REQUIRED_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "City",
    "Country"
]

ALL_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "District",
    "State",
    "Region",
    "Country"
]

BZK_ADDRESS_FIELDS = [
    'ApplicantCurrentAddress',
    'VictimBirthPlace',
    'VictimCurrentAddress',     
    'ApplicantBirthPlace',
    'VictimDeathPlace'
]

/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


## Set hardware accelerator


In [3]:
# List available hardware accelerators for PyTorch
import torch
available_accelerators = []
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
        available_accelerators.append(torch.device(f'cuda:{i}'))
elif torch.accelerator.is_available(): # Support other hardware accelators
    available_accelerators.append(torch.accelerator.current_accelerator())
else:
    print("WARNING: Running on CPU")
    device = torch.device('cpu')

CUDA - available devices:
  0: NVIDIA A100 80GB PCIe


/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [4]:
if available_accelerators:
    device = available_accelerators[0]
print(f"Torch version: {torch.__version__}, Device: {device}")

Torch version: 2.10.0+cu128, Device: cuda:0


## Load Dataset

Read all splits and concat them

In [5]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_merged : pd.DataFrame = pd.concat(
    [bzkopen_train, bzkopen_val, bzkopen_test], 
    keys=['train', 'val', 'test'], 
    names=['split', 'row']
)
display(bzkopen_merged.sample(5))

field                   FullAddress UnitNumber  \
split row                                                                     
train 445     VictimCurrentAddress           XXXXXX, Farenland 5        NaN   
      768      ApplicantBirthPlace               Kaunas, Litauen        NaN   
      535      ApplicantBirthPlace        Elisabethgrad/Russland        NaN   
      395  ApplicantCurrentAddress  Jaffo Str. Schtei Achajot 21        NaN   
      708         VictimBirthPlace                Kattowitz/O.S.        NaN   

          HouseNumber           StreetName Neighborhood           City  \
split row                                                                
train 445           5            Farenland          NaN            NaN   
      768         NaN                  NaN          NaN         Kaunas   
      535         NaN                  NaN          NaN  Elisabethgrad   
      395          21  Str. Schtei Achajot          NaN          Jaffo   
      708         NaN                  NaN          NaN      Kattowitz   

          District Region State   Country PostalCode  
split row                                             
train 445      NaN    NaN   NaN       NaN        NaN  
      768      NaN    NaN   NaN   Litauen        NaN  
      535      NaN    NaN   NaN  Russland        NaN  
      395      NaN    NaN   NaN       NaN        NaN  
      708      NaN   O.S.   NaN       NaN        NaN

## Create folds



In [6]:
N_FOLDS = 10

# Set shuffle seed for reproducibility
SEED=4841762


In [7]:
# Define paths for saving models and predictions
models_path = Path("models") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
models_path.mkdir(parents=True, exist_ok=True)
preds_path = Path("experiments_data") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
preds_path.mkdir(parents=True, exist_ok=True)

In [8]:
class Fold(NamedTuple):
    fold_idx: int
    train_data: pd.DataFrame
    val_data: pd.DataFrame
    models_path: Path
    preds_path: Path

In [9]:

shuffled_bzkopen = bzkopen_merged.sample(frac=1, random_state=SEED)

fold_indices = np.array_split(shuffled_bzkopen.index, N_FOLDS)

folds = []

for fold_idx, val_indices in enumerate(fold_indices):
    train_indices = shuffled_bzkopen.index.difference(val_indices)
    fold_models_path = models_path / f"fold_{fold_idx}"
    fold_models_path.mkdir(parents=True, exist_ok=True)
    fold_preds_path = preds_path / f"fold_{fold_idx}"
    fold_preds_path.mkdir(parents=True, exist_ok=True)
    folds.append(Fold(
        fold_idx, 
        shuffled_bzkopen.loc[train_indices], 
        shuffled_bzkopen.loc[val_indices],
        fold_models_path, 
        fold_preds_path
    ))
display(pd.DataFrame(
    [[len(x) for x in [fold.val_data, fold.train_data]] for fold in folds], 
    columns=['val_fold_size', 'train_fold_size'])
)

,val_fold_size,train_fold_size
0,108,967
1,108,967
2,108,967
3,108,967
4,108,967
5,107,968
6,107,968
7,107,968
8,107,968
9,107,968


# Train Spacy NER models

These models are used for the pattern example matching strategy so they will be reused over multiple different experiments. Therefore it is useful to train them in advance.

In [10]:
for fold in tqdm(folds, unit="fold"):
    output_dir = fold.models_path / 'ner_bzk'
    if output_dir.exists():
        print(f"Fold {fold.fold_idx} already has a trained model, skipping training.")
        continue
    train_ner.train(
        n_iter=30,
        output_dir=output_dir,
        train_df=fold.train_data,
        val_df=fold.val_data
    )

  0%|          | 0/10 [00:00<?, ?fold/s]

Fold 0 already has a trained model, skipping training.
Fold 1 already has a trained model, skipping training.
Fold 2 already has a trained model, skipping training.
Fold 3 already has a trained model, skipping training.
Fold 4 already has a trained model, skipping training.
Fold 5 already has a trained model, skipping training.
Fold 6 already has a trained model, skipping training.
Fold 7 already has a trained model, skipping training.
Fold 8 already has a trained model, skipping training.
Fold 9 already has a trained model, skipping training.


In [11]:
# Metrics on the required entities (Country, City, StreetName, HouseNumber)
required_results = OrderedDict()
specific_results = OrderedDict()


class FoldEvaluator:
    def __init__(self, model_name : str, supported_entities : list[str]):
        # Metrics on the required entities (Country, City, StreetName, HouseNumber)
        self._required_metric_records = None
        self._specific_metric_records = None
        self.required_metrics = None
        self.specific_metrics = None
        self.model_name = model_name
        self.supported_entities = supported_entities
    
    def __enter__(self):
        self.required_metrics = None
        self.specific_metrics = None
        self._required_metric_records = []
        self._specific_metric_records = []
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        global required_results, specific_results
        self.required_metrics = pd.DataFrame(self._required_metric_records).mean()
        self.specific_metrics = pd.DataFrame(self._specific_metric_records).mean()
        required_results[self.model_name] = self.required_metrics
        specific_results[self.model_name] = self.specific_metrics
        print(f"{self.model_name} - Metrics for Country City Street House:")
        display(self.required_metrics)
        print(f"{self.model_name} - Specific Metrics:")
        display(self.specific_metrics.unstack(level=0))
        self._required_metric_records = None
        self._specific_metric_records = None

    def _record_predictions(self, fold : Fold, preds : list[dict]):
        preds_df = pd.DataFrame(preds)
        fold_metrics = compare_preds(preds_df, fold.val_data, target_columns=REQUIRED_ENTITIES)
        self._required_metric_records.append(pd.Series(fold_metrics))
        specific_fold_metrics = OrderedDict()
        for entity in self.supported_entities:
            entity_metrics = compare_preds(preds_df, fold.val_data, target_columns=[entity])
            specific_fold_metrics[entity] = pd.Series(entity_metrics)
        for bzk_field in BZK_ADDRESS_FIELDS:
            mask = fold.val_data['field'] == bzk_field
            field_metrics = compare_preds(preds_df[mask.reset_index(drop=True)], fold.val_data[mask], target_columns=REQUIRED_ENTITIES)
            specific_fold_metrics[bzk_field] = pd.Series(field_metrics)
        self._specific_metric_records.append(pd.concat(specific_fold_metrics, names=['Field/Entity', 'Metric']))

    def load_preds_and_skip(self, folds : list[Fold]) -> list[Fold]:
        """
        For each fold, load its predictions if available on disk.
        Returns the folds for which predictions were not loaded
        """
        unloaded_folds = []
        for fold in folds:
            preds_file = fold.preds_path / self.model_name / "preds.json"
            if preds_file.exists():
                print(f"Fold {fold.fold_idx} already has predictions for {self.model_name}, loading from file.")
                with open(preds_file, "r") as f:
                    preds = json.load(f)
                self._record_predictions(fold, preds)
            else:
                unloaded_folds.append(fold)
        return unloaded_folds
            
    
    def run_on_fold(self, fold : Fold, model):
        assert self._required_metric_records is not None and self._specific_metric_records is not None, "FoldMetricRecorder must be used as a context manager"
        preds_file = fold.preds_path / self.model_name / "preds.json"
        preds_file.parent.mkdir(parents=True, exist_ok=True)
        preds = model.parse_addresses(fold.val_data['FullAddress'].tolist())
        with open(preds_file, "w") as f:
            json.dump(preds, f, indent=4)
        self._record_predictions(fold, preds)
    

In [12]:
with FoldEvaluator("libpostal", libpostal_client.LIBPOSTAL_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = libpostal_client.LibpostalClient()
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        model.close()

Fold 0 already has predictions for libpostal, loading from file.
Fold 1 already has predictions for libpostal, loading from file.
Fold 2 already has predictions for libpostal, loading from file.
Fold 3 already has predictions for libpostal, loading from file.
Fold 4 already has predictions for libpostal, loading from file.
Fold 5 already has predictions for libpostal, loading from file.
Fold 6 already has predictions for libpostal, loading from file.
Fold 7 already has predictions for libpostal, loading from file.
Fold 8 already has predictions for libpostal, loading from file.
Fold 9 already has predictions for libpostal, loading from file.
libpostal - Metrics for Country City Street House:


accuracy                     0.687913
precision                    0.637836
recall                       0.419072
f1                           0.505555
accuracy_with_tol_1          0.698390
accuracy_with_tol_2          0.706533
accuracy_with_tol_3          0.721655
accuracy_with_tol_4          0.742580
average_levenshtein          2.488696
average_similarity           0.728394
average_levenshtein_match    3.200120
average_similarity_match     0.933376
no_match_rate                0.219784
dtype: float64

libpostal - Specific Metrics:


Field/Entity,HouseNumber,StreetName,City,State,Country,PostalCode,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
Metric,,,,,,,,,,,
accuracy,0.869756,0.771218,0.319064,0.948884,0.791615,0.468354,0.529093,0.809698,0.598449,0.801865,0.820525
precision,0.748952,0.533138,0.615605,0.667500,0.740418,0.033333,0.639412,0.665810,0.621782,0.685237,0.435000
recall,0.679602,0.524948,0.299415,0.468452,0.317047,0.016667,0.411403,0.408604,0.432618,0.433710,0.502778
f1,0.712181,0.528305,0.401641,0.536922,0.441058,0.022222,0.500029,0.501249,0.509248,0.528823,0.455229
accuracy_with_tol_1,0.881862,0.774013,0.336743,0.951670,0.800943,0.468354,0.540611,0.813009,0.619634,0.810099,0.832192
accuracy_with_tol_2,0.906057,0.774013,0.337677,0.952596,0.808385,0.473001,0.553091,0.813009,0.634656,0.810694,0.855774
accuracy_with_tol_3,0.923728,0.780512,0.348849,0.955400,0.833532,0.473001,0.578030,0.830172,0.652770,0.816639,0.866380
accuracy_with_tol_4,0.950727,0.793510,0.374879,0.976791,0.851203,0.475805,0.615146,0.846629,0.664811,0.828181,0.872819
average_levenshtein,0.582321,2.770093,5.511111,0.272300,1.091260,0.501237,3.955788,1.397142,3.355475,1.439254,1.155406


In [13]:
with FoldEvaluator("deepparse", deepparse.DEEPPARSE_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = deepparse.DeepParseParser(device=device)
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for deepparse, loading from file.
Fold 1 already has predictions for deepparse, loading from file.
Fold 2 already has predictions for deepparse, loading from file.
Fold 3 already has predictions for deepparse, loading from file.
Fold 4 already has predictions for deepparse, loading from file.
Fold 5 already has predictions for deepparse, loading from file.
Fold 6 already has predictions for deepparse, loading from file.
Fold 7 already has predictions for deepparse, loading from file.
Fold 8 already has predictions for deepparse, loading from file.
Fold 9 already has predictions for deepparse, loading from file.
deepparse - Metrics for Country City Street House:


accuracy                     0.570431
precision                    0.321375
recall                       0.271941
f1                           0.294381
accuracy_with_tol_1          0.577862
accuracy_with_tol_2          0.593216
accuracy_with_tol_3          0.609724
accuracy_with_tol_4          0.630428
average_levenshtein          3.934809
average_similarity           0.648879
average_levenshtein_match    5.110830
average_similarity_match     0.841506
no_match_rate                0.229102
dtype: float64

deepparse - Specific Metrics:


Field/Entity,HouseNumber,City,Country,PostalCode,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
Metric,,,,,,,,,
accuracy,0.864183,0.253037,0.692073,0.922793,0.352289,0.763075,0.443397,0.707778,0.748977
precision,0.834101,0.318456,0.272803,0.025000,0.309011,0.414742,0.313204,0.328656,0.277089
recall,0.656874,0.255519,0.079102,0.016667,0.230186,0.412114,0.265327,0.311501,0.390278
f1,0.733792,0.283273,0.118521,0.020000,0.263626,0.412542,0.287069,0.319081,0.315471
accuracy_with_tol_1,0.886483,0.259536,0.692999,0.928375,0.365328,0.765249,0.458623,0.711914,0.748977
accuracy_with_tol_2,0.939529,0.264183,0.694860,0.939521,0.389361,0.765249,0.506893,0.712555,0.761488
accuracy_with_tol_3,0.963699,0.271625,0.725554,0.948840,0.419085,0.775304,0.535099,0.716106,0.770655
accuracy_with_tol_4,0.976739,0.292108,0.763690,0.956308,0.449611,0.787457,0.560573,0.732547,0.775655
average_levenshtein,0.453141,6.726912,1.825675,0.540247,6.415092,1.823489,5.038260,2.378246,2.282143


In [ ]:
with FoldEvaluator("xlm-roberta-large", token_classifiers.FIGHTING_CRIME_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = token_classifiers.TokenClassifierAddressParser(
            model_name="hm-haitham/xlm-roberta-large-address-parser", 
            device=device,
            aggregation_strategy="average"
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for xlm-roberta-large, loading from file.
Fold 1 already has predictions for xlm-roberta-large, loading from file.
Fold 2 already has predictions for xlm-roberta-large, loading from file.
Fold 3 already has predictions for xlm-roberta-large, loading from file.
Fold 4 already has predictions for xlm-roberta-large, loading from file.
Fold 5 already has predictions for xlm-roberta-large, loading from file.
Fold 6 already has predictions for xlm-roberta-large, loading from file.
Fold 7 already has predictions for xlm-roberta-large, loading from file.
Fold 8 already has predictions for xlm-roberta-large, loading from file.
Fold 9 already has predictions for xlm-roberta-large, loading from file.
xlm-roberta-large - Metrics for Country City Street House:


accuracy                     0.805393
precision                    0.709732
recall                       0.657383
f1                           0.682448
accuracy_with_tol_1          0.816312
accuracy_with_tol_2          0.822359
accuracy_with_tol_3          0.833511
accuracy_with_tol_4          0.848875
average_levenshtein          1.430642
average_similarity           0.840967
average_levenshtein_match    1.589168
average_similarity_match     0.933927
no_match_rate                0.099544
dtype: float64

xlm-roberta-large - Specific Metrics:


Field/Entity,Country,State,City,StreetName,HouseNumber,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
Metric,,,,,,,,,,
accuracy,0.909770,0.925632,0.517272,0.874507,0.920024,0.706104,0.874314,0.755926,0.875167,0.880260
precision,0.799440,0.287460,0.599839,0.746411,0.844150,0.717041,0.699505,0.732876,0.707772,0.555736
recall,0.795060,0.227500,0.520498,0.741961,0.799598,0.648604,0.619593,0.693563,0.655717,0.772222
f1,0.795242,0.250404,0.557059,0.743500,0.821117,0.680719,0.653286,0.711718,0.680419,0.635377
accuracy_with_tol_1,0.922785,0.932139,0.519133,0.886587,0.936743,0.725455,0.874314,0.780115,0.878169,0.889426
accuracy_with_tol_2,0.927440,0.934000,0.520059,0.886587,0.955348,0.732211,0.874314,0.798822,0.879505,0.901937
accuracy_with_tol_3,0.937660,0.947015,0.533091,0.894946,0.968345,0.752802,0.884369,0.811537,0.883510,0.904210
accuracy_with_tol_4,0.958143,0.964694,0.550762,0.900537,0.986059,0.777638,0.896018,0.828704,0.892113,0.906483
average_levenshtein,0.419514,0.410012,3.989339,1.055114,0.258602,2.176361,0.960009,1.760983,0.887042,0.889297


In [15]:
with FoldEvaluator("SpaCy", supported_entities=ALL_ENTITIES) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        for fold in tqdm(folds_to_process, unit="fold"):
            model = train_ner.SpaCyAddressParser(model_dir=fold.models_path / 'ner_bzk')
            evaluator.run_on_fold(fold, model)

  0%|          | 0/10 [00:00<?, ?fold/s]

SpaCy - Metrics for Country City Street House:


accuracy                     0.889339
precision                    0.853173
recall                       0.814881
f1                           0.833505
accuracy_with_tol_1          0.898163
accuracy_with_tol_2          0.906304
accuracy_with_tol_3          0.911193
accuracy_with_tol_4          0.917709
average_levenshtein          0.891310
average_similarity           0.912928
average_levenshtein_match    0.953698
average_similarity_match     0.970353
no_match_rate                0.059283
dtype: float64

SpaCy - Specific Metrics:


Field/Entity,HouseNumber,StreetName,Neighborhood,City,District,State,Region,Country,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
Metric,,,,,,,,,,,,,
accuracy,0.902328,0.878202,0.893129,0.815040,0.931187,0.950684,0.000000,0.961786,0.816595,0.976542,0.809389,0.954850,0.898707
precision,0.816794,0.807996,0.511732,0.859895,0.695310,0.736667,0.000000,0.938430,0.846199,0.949213,0.811590,0.905199,0.656558
recall,0.778129,0.729742,0.332692,0.839322,0.719165,0.413333,0.000000,0.902195,0.778141,0.936730,0.753509,0.898007,0.923611
f1,0.796591,0.766370,0.379130,0.849330,0.696681,0.484380,0.000000,0.919598,0.810604,0.942381,0.781103,0.901412,0.748309
accuracy_with_tol_1,0.924628,0.880054,0.893129,0.823390,0.931187,0.950684,0.000000,0.964581,0.834684,0.980032,0.818805,0.958399,0.898707
accuracy_with_tol_2,0.950692,0.881906,0.893129,0.827103,0.932113,0.954396,0.000000,0.965516,0.847567,0.980032,0.839960,0.958399,0.912516
accuracy_with_tol_3,0.958134,0.885618,0.894055,0.830841,0.933048,0.956265,0.000000,0.970180,0.855044,0.985032,0.849780,0.960743,0.912516
accuracy_with_tol_4,0.964642,0.894929,0.895916,0.839218,0.940498,0.983255,0.000000,0.972049,0.869205,0.985032,0.857722,0.962605,0.915016
average_levenshtein,0.456983,1.250511,1.102302,1.589339,0.568051,0.255858,0.685575,0.268406,1.426396,0.157555,1.588322,0.343311,0.956207


In [ ]:
supported_entities = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "State",
    "District",
    "Country"
]

prompt0_template = llm_parsers.JsonDictPromptTemplate(Path("prompts/prompt_0.txt").read_text())

with FoldEvaluator("Qwen3.5-9B-prompt0-0shot", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.QwenAddressParsingModel(
            model_name="Qwen/Qwen3.5-9B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt0_template
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 1 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 2 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 3 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 4 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 5 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 6 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 7 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 8 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 9 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Qwen3.5-9B-prompt0-0shot - Metrics for Country City Street House:


accuracy                     0.895360
precision                    0.859612
recall                       0.818855
f1                           0.838674
accuracy_with_tol_1          0.901179
accuracy_with_tol_2          0.906525
accuracy_with_tol_3          0.916295
accuracy_with_tol_4          0.927466
average_levenshtein          0.718092
average_similarity           0.919957
average_levenshtein_match    0.762734
average_similarity_match     0.975778
no_match_rate                0.057195
dtype: float64

Qwen3.5-9B-prompt0-0shot - Specific Metrics:


Field/Entity,HouseNumber,StreetName,Neighborhood,City,State,District,Country,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
Metric,,,,,,,,,,,,
accuracy,0.952605,0.914417,0.893977,0.786068,0.933982,0.866978,0.928349,0.857273,0.924974,0.841126,0.928818,0.940471
precision,0.879931,0.817927,0.366535,0.863233,0.490490,0.309574,0.889508,0.861688,0.895255,0.804774,0.898396,0.781429
recall,0.889203,0.819884,0.392033,0.796758,0.905278,0.373518,0.792191,0.843141,0.762941,0.797302,0.790922,0.883333
f1,0.884422,0.818637,0.376215,0.828443,0.633103,0.332916,0.836703,0.852241,0.820466,0.800856,0.840607,0.816666
accuracy_with_tol_1,0.955400,0.925606,0.895829,0.789780,0.936777,0.869773,0.933930,0.869735,0.926290,0.848150,0.930374,0.940471
accuracy_with_tol_2,0.973053,0.926532,0.905140,0.790715,0.939573,0.871634,0.935800,0.875875,0.926290,0.873366,0.931763,0.940471
accuracy_with_tol_3,0.985125,0.930253,0.906066,0.802821,0.943294,0.875363,0.946980,0.891148,0.938845,0.889169,0.936089,0.945244
accuracy_with_tol_4,0.991641,0.941433,0.908844,0.813058,0.947015,0.889313,0.963733,0.910519,0.945403,0.896015,0.944137,0.947516
average_levenshtein,0.162600,0.710038,0.867792,1.636951,0.479751,1.002838,0.362781,0.977793,0.536228,0.999199,0.491100,0.452137


In [ ]:
prompt_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/prompt_optuna_best.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35


with FoldEvaluator("Qwen3.5-9B-best-from-optuna", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.QwenAddressParsingModel(
            model_name="Qwen/Qwen3.5-9B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            pattern_similarity = llm_parsers.NERPatternSimilarExamples(
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                model_dir = fold.models_path / "ner_bzk"
            )
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            hybrid_similarity = llm_parsers.HybridSimilarExamples(
                pattern_strategy=pattern_similarity,
                embedding_strategy=embedding_similarity,
                num_examples=n_examples,
                pool_size=n_examples
            )
            model.example_strategy = hybrid_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del hybrid_similarity, embedding_similarity, pattern_similarity
        del model

Fold 0 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 1 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 2 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 3 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 4 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 5 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 6 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 7 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 8 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 9 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Qwen3.5-9B-best-from-optuna - Metrics for Country City Street House:


accuracy                     0.955136
precision                    0.927202
recall                       0.930435
f1                           0.928792
accuracy_with_tol_1          0.959320
accuracy_with_tol_2          0.960951
accuracy_with_tol_3          0.966295
accuracy_with_tol_4          0.970020
average_levenshtein          0.312188
average_similarity           0.966598
average_levenshtein_match    0.319164
average_similarity_match     0.986535
no_match_rate                0.020217
dtype: float64

Qwen3.5-9B-best-from-optuna - Specific Metrics:


Field/Entity,HouseNumber,StreetName,Neighborhood,City,Country,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
Metric,,,,,,,,,,
accuracy,0.971166,0.963768,0.914495,0.913525,0.972084,0.928175,0.985529,0.927898,0.972624,0.980833
precision,0.932043,0.936583,0.510210,0.919659,0.932656,0.924335,0.959860,0.911838,0.933448,0.918889
recall,0.934631,0.925679,0.644505,0.925086,0.947195,0.924097,0.969451,0.908924,0.942576,0.977778
f1,0.933210,0.930753,0.566442,0.922332,0.939295,0.924152,0.964301,0.910234,0.937756,0.938562
accuracy_with_tol_1,0.974888,0.965620,0.916346,0.919107,0.977665,0.933486,0.988768,0.933623,0.976308,0.980833
accuracy_with_tol_2,0.979543,0.966554,0.919124,0.919107,0.978600,0.938014,0.988768,0.936401,0.976308,0.980833
accuracy_with_tol_3,0.990689,0.971184,0.919124,0.920976,0.982330,0.949307,0.993768,0.942193,0.978722,0.980833
accuracy_with_tol_4,0.994418,0.974913,0.925640,0.923771,0.986976,0.957500,0.993768,0.945330,0.980904,0.980833
average_levenshtein,0.106006,0.336345,0.754898,0.671452,0.134952,0.487113,0.067806,0.504158,0.190698,0.170000


In [ ]:
all_experiments = pd.DataFrame({k : v[["f1", "precision", "recall", "accuracy"]] for k, v in required_results.items()}).T

all_experiments_styled = (
    all_experiments.style.format("{:.2f}").apply(
        lambda s: ['font-weight: bold' if v == s.max() else '' for v in s],
        axis=0
    )
)
display(all_experiments_styled)

,f1,precision,recall,accuracy
libpostal,0.51,0.64,0.42,0.69
deepparse,0.29,0.32,0.27,0.57
xlm roberta large,0.68,0.71,0.66,0.81
SpaCy,0.83,0.85,0.81,0.89
Qwen3.5 9B prompt0 0shot,0.84,0.86,0.82,0.90
Qwen3.5 9B best from optuna,0.93,0.93,0.93,0.96


In [19]:
# Subset of experiments selected for inclusion in the paper.
selected_experiments = all_experiments

# Sanity check that the selected experiments include all the best results
assert all(all_experiments[c].argmax() == selected_experiments[c].argmax() for c in all_experiments.columns), "Selected experiments must have the same best metrics as all experiments for the required entities"

selected_experiments_styled = (
    selected_experiments.style.format("{:.2f}").apply(
        lambda s: ['font-weight: bold' if v == s.max() else '' for v in s],
        axis=0
    )
)

display(selected_experiments_styled)

,f1,precision,recall,accuracy
libpostal,0.51,0.64,0.42,0.69
deepparse,0.29,0.32,0.27,0.57
xlm roberta large,0.68,0.71,0.66,0.81
SpaCy,0.83,0.85,0.81,0.89
Qwen3.5 9B prompt0 0shot,0.84,0.86,0.82,0.90
Qwen3.5 9B best from optuna,0.93,0.93,0.93,0.96


In [ ]:
print(
    selected_experiments_styled.to_latex(hrules=True, convert_css=True)
      .replace("& f1 &", "& F_1 &")
)

\begin{tabular}{lrrrr}
\toprule
 & F_1 & precision & recall & accuracy \\
\midrule
libpostal & 0.51 & 0.64 & 0.42 & 0.69 \\
deepparse & 0.29 & 0.32 & 0.27 & 0.57 \\
xlm roberta large & 0.68 & 0.71 & 0.66 & 0.81 \\
SpaCy & 0.83 & 0.85 & 0.81 & 0.89 \\
Qwen3.5 9B prompt0 0shot & 0.84 & 0.86 & 0.82 & 0.90 \\
Qwen3.5 9B best from optuna & \bfseries 0.93 & \bfseries 0.93 & \bfseries 0.93 & \bfseries 0.96 \\
\bottomrule
\end{tabular}

